# Let's try some **regression**!!

## Simple Regression

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import src.machine_learning as ML


class GlyphClassifier(nn.Module):
    def __init__(self, resolution):
        super(GlyphClassifier, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.classifier = nn.Sequential(
            nn.Linear(128 * resolution[0]//8 * resolution[1]//8, 1024),
            nn.ReLU(),
            nn.Linear(1024, 1)
        )
 
    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)  # Flatten the output
        x = self.classifier(x)
        return x

In [ ]:
# Getting the dataset
dataset_file = 'data/simple-star.zip'
train_dataset = ML.GlyphDataset(dataset_file, resize=(128,128), split = "train")
test_dataset = ML.GlyphDataset(dataset_file, resize=(128,128),split = 'test')
train_dataset.show()

# Assign the loaders 

train_loader = ML.create_loader(train_dataset, batch_size=64, shuffle = True)
test_loader = ML.create_loader(test_dataset, batch_size=64, shuffle = False)
ML.visualize_loader(train_loader,max_images=10,nrow=5)


In [ ]:
import torch
import matplotlib.pyplot as plt
import wandb

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

model = GlyphClassifier(resolution=(128, 128)).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)

# Training parameters
num_epochs = 50
losses = []
steps = 0  # Initialize step counter

# Initialising Weights&Biasis
wandb.init(
    project="glyph-regression",
    name="exp-1024neuron-128x128res-simplestar-logging steps",
    config={
        "architecture": "CNN-Glyph",
        "epochs": num_epochs,
        "batch_size": 64,
        "learning_rate": 0.0005,
        "loss_fn": "SmoothL1Loss",
        "optimizer": "Adam",
        "image_resolution": (128, 128),
        "regression": True
    }
)

# Watch model with Weights & Biases 
wandb.watch(model, log="all", log_freq=10)

for epoch in range(num_epochs):
    model.train()
    for images, values in train_loader:
        labels = torch.tensor(values, dtype=torch.float32).unsqueeze(1).to(device)  # Shape (batch, 1)

        images = images.to(device)
        optimizer.zero_grad()
        outputs = model(images).squeeze()
        loss = criterion(outputs, labels.squeeze())
        loss.backward()
        optimizer.step()
        
        losses.append(loss.item())
        steps += 1  # Increment step counter
        
        # Log at each step
        wandb.log({
            "step": steps,
            "train_loss": loss.item(),
            "epoch": epoch + 1  # Still track epoch for reference
        })
        
        print(f"Step {steps} - Epoch {epoch+1}/{num_epochs} - Loss: {loss.item():.4f}")

ML.plot_training_loss(losses)

In [ ]:
import numpy as np 

model.eval()
predictions = []
ground_truths = []

with torch.no_grad():
    for images, _ in test_loader:
        images = images.to(device)
        outputs = model(images).squeeze()
        predictions.extend(outputs.cpu().numpy())
        ground_truths.extend(labels.cpu().numpy())

predictions = np.array(predictions)
ground_truths = np.array(ground_truths)

mse = np.mean((predictions - ground_truths)**2)
mae = np.mean(np.abs(predictions - ground_truths))

print(f"Regression MSE: {mse:.4f}")
print(f"Regression MAE: {mae:.4f}")
wandb.log({
    "test_mse": mse,
    "test_mae": mae
})

wandb.finish()

## Binned Regression

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import src.machine_learning as ML

class GlyphClassifier(nn.Module):
    def __init__(self, NUM_bins, resolution):
        super(GlyphClassifier, self).__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2)
        )
        self.classifier = nn.Sequential(
            nn.Linear(128 * resolution[0]//8 * resolution[1]//8, 256),
            nn.ReLU(),
            nn.Linear(256, NUM_bins)
        )

    def forward(self, x):
        x = self.features(x)
        x = x.view(x.size(0), -1)  
        x = self.classifier(x)     
        return x


NameError: name 'nn' is not defined

In [ ]:
# Getting the dataset
dataset_file = 'data/simple-star.zip'
train_dataset = ML.GlyphDataset(dataset_file, resize=(128,128), split = "train")
test_dataset = ML.GlyphDataset(dataset_file, resize=(128,128),split = 'test')
train_dataset.show()

# Assign the loaders 

train_loader = ML.create_loader(train_dataset, batch_size=64, shuffle = True)
test_loader = ML.create_loader(test_dataset, batch_size=64, shuffle = False)
ML.visualize_loader(train_loader,max_images=10,nrow=5)


In [ ]:
# Setup
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

num_bins = 20
bin_centers = torch.linspace(2.5, 97.5, num_bins).to(device)  # Center of bins

# Model
model = GlyphClassifier(resolution=(128, 128), NUM_bins=num_bins).to(device)

# Loss and optimizer
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.0005)

# Initialize Weights & Biases
wandb.init(
    project="glyph-regression",
    name="exp-SimpleStar-128x128-20bins-BinnedRegression",
    config={
        "architecture": "CNN-Glyph",
        "epochs": 10,
        "batch_size": 64,
        "learning_rate": 0.0005,
        "loss_fn": "MSELoss",
        "optimizer": "Adam",
        "image_resolution": (128, 128),
        "regression": True,
        "num_bins": num_bins,
    }
)

wandb.watch(model, log="all", log_freq=10)

# --- Training loop ---
num_epochs = 20
losses = []
global_step = 0

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for images, values in train_loader:
        images = images.to(device)
        values = torch.tensor(values, dtype=torch.float32, device=device)

        optimizer.zero_grad()

        outputs = model(images)
        probabilities = torch.softmax(outputs, dim=1)  # manually here!
        predictions = torch.sum(probabilities * bin_centers, dim=1)
        loss = criterion(predictions, values)
        
        loss.backward()
        optimizer.step()

        # Track loss
        running_loss += loss.item()
        losses.append(loss.item())

        wandb.log({
            "train_loss_step": loss.item(),
            "global_step": global_step
        })

        if global_step % 100 == 0:
            print(f"Step {global_step}: Loss = {loss.item():.4f}")

        global_step += 1

    avg_loss = running_loss / len(train_loader)
    print(f"Epoch {epoch+1}/{num_epochs} - Average Loss: {avg_loss:.4f}")

# Plot training curve
ML.plot_training_loss(losses, title="Training Loss over Steps", xlabel="Steps", ylabel="MSE Loss")



In [ ]:
import numpy as np

# Evaluation mode
model.eval()
predictions = []
ground_truths = []

with torch.no_grad():
    for images, values in test_loader:  # Assuming `test_loader` yields images and values (not labels)
        images = images.to(device)
        values = torch.tensor(values, dtype=torch.float32, device=device)  # Ensure values are on the correct device
        
        # Get the model outputs
        logits = model(images)
        
        # Apply softmax to get probabilities
        probabilities = F.softmax(logits, dim=1)
        
        # Compute the predicted value using weighted sum of probabilities
        batch_predictions = torch.sum(probabilities * bin_centers, dim=1)  # shape: [batch_size]
        
        # Store the predictions and ground truths
        predictions.extend(batch_predictions.cpu().numpy())
        ground_truths.extend(values.cpu().numpy())  # ground truths are in the original scale [0, 100]

# Convert to numpy arrays and scale if necessary (but values are already in the correct range now)
predictions = np.array(predictions)
ground_truths = np.array(ground_truths)

# Calculate MSE and MAE
mse = np.mean((predictions - ground_truths)**2)
mae = np.mean(np.abs(predictions - ground_truths))

# Print the results
print(f"Test MSE: {mse:.4f}")
print(f"Test MAE: {mae:.4f}")

# Log results to Weights & Biases
wandb.log({
    "test_mse": mse,
    "test_mae": mae
})

wandb.finish()
